# Stage 4 – Class Weights & Advanced Fine-Tuning

In [15]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2, 3, 4])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(class_id): float(weight)
    for class_id, weight in zip(classes, weights)
}

print("Class Weights:")
for class_id, weight in class_weights.items():
    print(f"Class {class_id}: {weight:.3f}")

Class Weights:
Class 0: 0.622
Class 1: 3.033
Class 2: 1.124
Class 3: 1.166
Class 4: 0.761


In [16]:
from tensorflow import keras
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

model_cw = keras.models.load_model(
    "/kaggle/working/best_efficientnet_b0_final.keras"
)

model_cw.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_cw = [
    ModelCheckpoint(
        "/kaggle/working/best_efficientnet_b0_classweights.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

history_cw = model_cw.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weights,
    callbacks=callbacks_cw
)

Epoch 1/8
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.8186 - loss: 0.5943
Epoch 1: val_loss improved from None to 0.65806, saving model to /kaggle/working/best_efficientnet_b0_classweights.keras

Epoch 1: finished saving model to /kaggle/working/best_efficientnet_b0_classweights.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 54s 209ms/step - accuracy: 0.8095 - loss: 0.5814 - val_accuracy: 0.7558 - val_loss: 0.6581 - learning_rate: 1.0000e-05
Epoch 2/8
140/141 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8245 - loss: 0.5610
Epoch 2: val_loss improved from 0.65806 to 0.65794, saving model to /kaggle/working/best_efficientnet_b0_classweights.keras

Epoch 2: finished saving model to /kaggle/working/best_efficientnet_b0_classweights.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.8029 - loss: 0.5731 - val_accuracy: 0.7531 - val_loss: 0.6579 - learning_rate: 1.0000e-05
Epoch 3/8
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8052 - loss: 0.6041
Epoch 3: val_loss im

In [17]:

import numpy as np
from tensorflow import keras
from sklearn.metrics import classification_report, accuracy_score, f1_score

                                                       
best_cw = keras.models.load_model(
    "/kaggle/working/best_efficientnet_b0_classweights.keras"
)

                           
y_prob_cw = best_cw.predict(val_ds, verbose=1)
y_pred_cw = np.argmax(y_prob_cw, axis=1)

print("Validation Accuracy:", accuracy_score(y_val, y_pred_cw))
print("Macro F1-score:", f1_score(y_val, y_pred_cw, average="macro"))

print(
    classification_report(
        y_val,
        y_pred_cw,
        labels=[0, 1, 2, 3, 4],
        target_names=[
            "No DR",
            "Mild",
            "Moderate",
            "Severe",
            "Proliferative DR"
        ],
        zero_division=0
    )
)


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 317ms/step
Validation Accuracy: 0.7612551159618008
Macro F1-score: 0.6023727797205293
                  precision    recall  f1-score   support

           No DR       0.95      0.96      0.95       361
            Mild       0.45      0.66      0.54        74
        Moderate       0.78      0.60      0.68       200
          Severe       0.35      0.54      0.42        39
Proliferative DR       0.48      0.37      0.42        59

        accuracy                           0.76       733
       macro avg       0.60      0.63      0.60       733
    weighted avg       0.78      0.76      0.77       733



In [18]:

from tensorflow import keras
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

                                        
model_exp = keras.models.load_model(
    "/kaggle/working/best_efficientnet_b0_final.keras"
)

                                          
base_model = model_exp.get_layer("efficientnetb0")
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

                                       
for layer in base_model.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

                                
model_exp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-6),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks_exp = [
    ModelCheckpoint(
        "/kaggle/working/best_efficientnet_b0_exp50.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-7,
        verbose=1
    )
]

history_exp = model_exp.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks_exp
)


Epoch 1/12
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.8277 - loss: 0.4500
Epoch 1: val_loss improved from None to 0.63959, saving model to /kaggle/working/best_efficientnet_b0_exp50.keras

Epoch 1: finished saving model to /kaggle/working/best_efficientnet_b0_exp50.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 53s 214ms/step - accuracy: 0.8182 - loss: 0.4749 - val_accuracy: 0.7626 - val_loss: 0.6396 - learning_rate: 5.0000e-06
Epoch 2/12
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.8341 - loss: 0.4445
Epoch 2: val_loss improved from 0.63959 to 0.62055, saving model to /kaggle/working/best_efficientnet_b0_exp50.keras

Epoch 2: finished saving model to /kaggle/working/best_efficientnet_b0_exp50.keras
141/141 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.8196 - loss: 0.4659 - val_accuracy: 0.7681 - val_loss: 0.6205 - learning_rate: 5.0000e-06
Epoch 3/12
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8244 - loss: 0.4543
Epoch 3: val_loss did not improve from 0.6205

In [19]:

import numpy as np
from tensorflow import keras
from sklearn.metrics import classification_report, f1_score

best_model = keras.models.load_model(
    "/kaggle/working/best_efficientnet_b0_exp50.keras"
)

y_prob = best_model.predict(val_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("Macro F1:", f1_score(y_val, y_pred, average="macro"))

print(classification_report(
    y_val,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    target_names=[
        "No DR",
        "Mild",
        "Moderate",
        "Severe",
        "Proliferative DR"
    ],
    zero_division=0
))


23/23 ━━━━━━━━━━━━━━━━━━━━ 13s 312ms/step
Macro F1: 0.6118013145728944
                  precision    recall  f1-score   support

           No DR       0.95      0.96      0.95       361
            Mild       0.49      0.45      0.47        74
        Moderate       0.73      0.71      0.72       200
          Severe       0.42      0.51      0.46        39
Proliferative DR       0.47      0.44      0.46        59

        accuracy                           0.78       733
       macro avg       0.61      0.62      0.61       733
    weighted avg       0.77      0.78      0.78       733

